In [ ]:
import ee
import geemap
import pandas as pd
import numpy as np

In [ ]:
try:
    ee.Initialize()
except: 
    ee.Authenticate()
    ee.Initialize()

# Spectral Change

This notebook utilizes Google Earth Engine to access Sentinel-2 to view NDVI trends over time. The temporal trends can be year-over-year or intrayear (within one year). You can download the resulting trends as TIFFs. You may also download the mean index value of a region for each date (image) in the image collection.

These workflows were used to produce the remote NDVI data for Kastning et al.

# Functions

In [ ]:
def maskS2clouds(image):
    """ Mask cloud and cirrus pixels from Sentinel-2 imagery using QA60 band.

        Parameters:
        image (Image): A single Image in an ImageCollection or standalone Image

        Returns:
        Image with masked features removed and original metadata

    """
    qa = image.select('QA60')

    # Bits 10 and 11 are clouds and cirrus
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11

    # Both flags should be set to zero, indicating clear conditions.
    mask = qa.bitwiseAnd(cloudBitMask).eq(0) \
        .And(qa.bitwiseAnd(cirrusBitMask).eq(0)) # performs bitwise AND operation between QA60 band and cloud bitmask

    return image.updateMask(mask) \
        .divide(10000) \
        .copyProperties(image, ['system:time_start']) 

def maskWater(image):
    ''' Mask out water using MODIS data.

        Returns: 
        Image with pixels where water_mask < 1. '''
    # Get water mask
    waterMask = (
        ee.ImageCollection('MODIS/006/MOD44W') 
        .filter(ee.Filter.date('2015-01-01', '2015-01-02')) 
        .select('water_mask') \
        .first()
    )
    
    return image.updateMask(waterMask.select('water_mask').lt(1))

def maskS2snow(image):
    ''' Mask snow from Sentinel-2 imagery with MSK_SNWPRB (snow probability mask).

        Returns: 
        Image with pixels where MSK_SNWPRB < 0.9%. '''
    
    mask = image.select('MSK_SNWPRB').lt(0.009)
    
    return image.updateMask(mask).copyProperties(image, ['system:time_start'])

def maskS2White(image):
    ''' Masks S2 white pixels to ensure all cloudy or snowy pixels are removed.

        Returns: 
        Image with pixels where grayscale is greater than or equal to 2000 are removed. '''

    # convert RGB values to grayscale
    grayscale = image.expression(
            '(.3 * 1e4 * R) + (.59 * 1e4 * G) + (.11 * 1e4 * B)', {
            'R': image.select('B4'),
            'G': image.select('B3'),
            'B': image.select('B2')
        })
    
    white_mask = grayscale.lte(2000)
    
    return image.updateMask(white_mask).copyProperties(image,['system:time_start'])

def calculateNoDataPercentage(image):
        """Add data on masked pixel percentage as a property

    Parameters:
    image (Image): A single Image in an ImageCollection or standalone Image

    Returns:
    Image with "nodata_percentage property" set

    """
        pixel_area = ee.Image.pixelArea()
        
        nodata_mask = (image.select('red')).eq(0) # choose any band
        nodata_area = pixel_area.updateMask(nodata_mask)
        
        nodata_area = nodata_area.reduceRegion(
              reducer=ee.Reducer.sum(),
              geometry=aoi,
              scale=30,
              maxPixels=1e9
            ).get('area')
        
        total_area = pixel_area.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=aoi,
            scale=30,
            maxPixels=1e9
            ).get('area')


        nodata_pixels = ee.Number(nodata_area)
        total_pixels = ee.Number(total_area)
        
        # Calculate percentage
        nodata_percentage = nodata_pixels.divide(total_pixels).multiply(100)
        
        # Add NoData percentage as a property
        return image.set('nodata_percentage', nodata_percentage)

def clp(image):
    '''Clips a single Image to a region of interest'''
    return image.clip(aoi)

def addNDVI(image):
  '''Adds NDVI band to each image (in an ImageCollection)

    Requires redBand and NIRband be defined prior to running. '''

  ndvi = image.normalizedDifference(['NIR', 'red']).rename('NDVI')

  return image.addBands(ndvi)


In [ ]:
def getS2scenes(start_year, end_year, start_month, end_month):
    ''' Build image collection with Sentinel-2 scenes and define redBand and NIRband for NDVI calculations.'''
    
    # rename S2 bands to standard names
    def _nameS2Bands(image):
        originalBands = ['B2', 'B3', 'B4', 'B8']
        namedBands = ['blue', 'green', 'red', 'NIR']
        return image.select(originalBands, namedBands)
    
    # Get Sentinel 2 harmonized images
    dataset = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                    .filter(ee.Filter.calendarRange(start_year, end_year, 'year'))
                    .filter(ee.Filter.calendarRange(start_month, end_month, 'month'))
                    #  Pre-filter to get less cloudy granules.
                    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
                    .filterBounds(aoi)
                    .map(clp)
                    .map(maskS2clouds)
                    .map(maskS2snow) 
                    .map(maskS2White)
                    .map(maskWater)
                    .map(_nameS2Bands)
    )

    return dataset

def annual_images(y):
    ''' Filters an image collection for a specific year (y) and date range (start and end month).
        Applies the chosen analysis (mean, min, max, or median) to calculate the statistics for the images in that year.
        
        Requires:
        index_collection, start_month, end_month, and analysis.
    
        Parameters: 
        y: a given year

        Returns:
        Statistcs over the image within the date range.
    '''
    # filter for given year
    range_year = ee.Filter.calendarRange(y, y, 'year')
    #filter for specified month range
    range_month = ee.Filter.calendarRange(start_month, end_month, 'month')

    # filter image collection by year and month and add a time band
    filtered_dataset = (index_collection
                        .filter(range_year)
                        .filter(range_month)
                        .map(lambda image: image.addBands(image.metadata('system:time_start').divide(3.154e10)))) # Needed for linear regression 
        
    # Print out the number of images in the filtered dataset
    num_images = filtered_dataset.size()
    
    # Choose the reducer based on the chosen analysis
    if analysis == 'mean':
        reducer = ee.Reducer.mean().combine( # calculates average value for each pixel across all images
            reducer2=ee.Reducer.stdDev(), # calculates standard deviation
            sharedInputs=True
        )

    elif analysis == 'min' or analysis == 'max':
        reducer = ee.Reducer.mean().combine( # calculates minimum and maximum values
            reducer2=ee.Reducer.minMax(),#
            sharedInputs=True
        )
    elif analysis == 'median':
        reducer = ee.Reducer.mean().combine( # calculates middle value for each pixel
            reducer2=ee.Reducer.median(),
            sharedInputs=True
        )

    # Use the combined reducer to get the statistics
    stats = filtered_dataset.reduce(reducer)
    return stats.set('year', y).set('num', num_images)
    

def intrayear(index_collection):
    ''' Filter Image Collection within a single year. Used for seasonal (one-year) trends.

        Function is used when start_year = end_year.

        Requires: 
        index_collection, start_month, end_month, start_year, and end_year.

        Parameters: 
        index_collection: an ImageCollection containing images with a chosen index

        Returns:
        filtered_dataset: Filtered Image Collection
    '''
    
    range_year = ee.Filter.calendarRange(start_year, end_year, 'year')
    range_month = ee.Filter.calendarRange(start_month, end_month, 'month')
    
    # add a time band with the year to each image 
    filtered_dataset = (index_collection
                        .filter(range_year)
                        .filter(range_month)
                        .map(lambda image: image.addBands(image.metadata('system:time_start').divide(3.154e10)))) # Needed for linear regression 
    
    num_images = filtered_dataset.size()

    return filtered_dataset.set('year', start_year).set('num', num_images)

def createTimeBand(image):   
    '''Adds a time band to the image using metadata'''
    return image.addBands(image.metadata('system:time_start').divide(3.154e10))

def meanNDVI(image): 
    ''' Calculates the mean NDVI within a specific region.
        
        Requires:
        ndvi, aoi

        Returns:
        Image with property 'ndvi_mean'. '''
    
    ndvi = image.select('NDVI')
    ndvi_mean = ndvi.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=10
    ).get('NDVI')

    ndvi_mean = ee.Number(ndvi_mean)
    
    return image.set('ndvi_mean', ndvi_mean)


# Build collection

### Define AOI and build image collection. Choose a cell to use coordinates, or upload a shapefile.

Option 1: Use coordinates

In [ ]:
lat, long = (69.5, -143.66) # North Slope, AK

aoi_point = ee.Geometry.Point([long, lat])
aoi = aoi_point.buffer(1000).bounds() # change buffer size for different area

Option 2: upload shp from local directory

In [ ]:
aoi = geemap.shp_to_ee('wt_veg/shapefiles/8100087910/8100087910_wt.shp')    

long = aoi.geometry().centroid(maxError=1).coordinates().get(0).getInfo()
lat = aoi.geometry().centroid(maxError=1).coordinates().get(1).getInfo()

# Build Image Collection

Sentinel-2 imagery (10-m) is available after 2019.

Enter inputs:
- start_year, end_year: filter image collection by months
- start_month, end_month: filter image collection by year

Enter thresholds for filtering data collection:
- cloud_threshold: percentage of cloudy pixels (0-100)
- threshold_nodata_percent: percentage of AOI that has no data (0-100)
   - *Tip: Larger AOIs and Landsat may require a larger threshold to acquire enough imagery*

In [ ]:
# Choose timespan and filters

# reminder that Sentinel is available beginning in 2019
start_year, end_year = 2019, 2025
start_month, end_month = 6, 8

# adjust thresholds
threshold_nodata_percent = 5
cloud_threshold = 5

In [ ]:
# build image collection

im_col = getS2scenes(start_year, end_year, start_month, end_month)

# filter for no data and add NDVI band to images
dataset = (
    im_col.map(calculateNoDataPercentage)
    .filter(ee.Filter.lte('nodata_percentage', threshold_nodata_percent))
    .map(addNDVI)
)

## Do analysis

Select your index and time frame.

- Year-over-year trend in peak greenness: NDVI max from June through August
- Year-over-year change in early season greenness: NDVI max in June (start and end month = 6)
- Seasonal, intra-year trend in greenness: NDVI max from June to August in just 2025 (start and end year = 2025)

Choosing 'max' for analysis is recommended with NDVI to avoid potential influence of snow.

In [ ]:
# Pick your index: NDVI 
index = 'NDVI'
# Choose 'mean', 'median', 'min', or 'max' for analysis
analysis = 'max'  

# image collection with one band for NDVI
index_collection = dataset.select(index)  

# Generate list of years
years = ee.List.sequence(start_year, end_year)

# Skip to different sections to:

1. view spectral trends on the map and download TIF
2. save and plot index values

# 1. View Trend

In [ ]:
# intrayear
if start_year == end_year:
    intrayear_collection = intrayear(index_collection)

    item = intrayear_collection.getInfo()
    print("Year:", item['properties']['year'], "Number of images:", item['properties']['num'])

    # Get linear fit to pixelwise trend of annual max NDVI
    trend = intrayear_collection.select(['system:time_start',
                                index
                                ]).reduce(ee.Reducer.linearFit())

# year-over-year
else:

    # Map over years to get yearly statistics
    yearwise_ndvi = years.map(annual_images)

    for item in yearwise_ndvi.getInfo():
        print("Year:", item['properties']['year'], "Number of images:", item['properties']['num'])

        yearCompCol = ee.ImageCollection.fromImages(yearwise_ndvi)

        # Get linear fit to pixelwise trend of annual max NDVI
        trend = yearCompCol.select(['system:time_start_mean',
                                    f'{index}_{analysis}'
                                    ]).reduce(ee.Reducer.linearFit())



### Define colorbar min and max and choose color palette.
Find the min and max values of the trend, which will be used as the min and max values of the colorbar.

In [ ]:
scale = trend.select('scale')

tMin = scale.reduceRegion(
    reducer=ee.Reducer.min(),
    geometry=aoi,
    scale=500,  # Adjust the scale depending on your resolution
    maxPixels=1e9
).getInfo()

tMax = scale.reduceRegion(
    reducer=ee.Reducer.max(),
    geometry=aoi,
    scale=500,  # Adjust the scale depending on your resolution
    maxPixels=1e9
).getInfo()

print(tMin)
print(tMax)

If the trend is only positive, make c_min = 0. 
If not, I recommend having the c_min and c_max be the same magnitude away from 0. 

Choose color palette where blue and red are positive and negative trends, respectively.

In [ ]:
# determine colorbar min and max
c_min = -0.05
c_max = 0.05

# palette
c_pal = ['red','white','blue'] # positive and negative
# c_pal = ['white','blue'] # positive

In [ ]:
# view trend
Map = geemap.Map(center=[lat, long], zoom=11)

Map.addLayer(trend.select('scale'),
              {'min': c_min, 'max': c_max,
             'palette': c_pal
             },
  'trend')

# add colorbar for trend
Map.add_colorbar_branca(colors=c_pal, vmin= c_min, vmax = c_max, layer_name='trend')

Map

download trend as TIFF

In [ ]:
geemap.download_ee_image(trend.select('scale'), 'NorthSlope-DGUP.tif', scale=10, region=aoi, crs='EPSG:3995')

# 2. Begin here to download NDVI data

Calculate mean NDVI of the entire area for each image date in the Image Collection and convert to a data frame.

For future plots, save data for water track AOI and intertrack AOI.

In [ ]:
# average NDVI of entire aoi per image
ndvi_collection = index_collection.map(meanNDVI)

image_list = ndvi_collection.getInfo()
data = []

# add date and mean NDVI to data frame
for img in image_list['features']:
    properties = img['properties']
    data.append({
            'date': ee.Date(properties['system:time_start']).format('YYYY-MM-dd').getInfo(),
            'ndvi_mean': properties.get('ndvi_mean')
        })
    
df = pd.DataFrame(data, columns=['date', 'ndvi_mean'])

In [ ]:
# save csv 
df.to_csv('wt_veg/8100087910_ndvi_19-25.csv')